# Laboratorio 17 — Series Temporales: Correlación y ARIMAX
**Curso:** Minería de Datos (EIN132A25)

## Objetivos
- Manipular índices temporales y descomponer una serie en tendencia/estacionalidad/residuo
- Diagnosticar estacionariedad (ADF) y diferenciar
- Interpretar **ACF / PACF** para elegir `(p, q)` y **CCF** para ver correlación con exógenas
- Ajustar progresivamente **AR → ARMA → ARIMA → SARIMA → SARIMAX** sobre el mismo dataset
- Validar con walk‑forward y métricas (MAE, RMSE, MAPE)

## Dataset
Demanda diaria sintética de un retail (≈3 años) con tendencia, estacionalidad semanal y anual, ruido y una **exógena observable** (temperatura). Reproducible (`seed=42`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, ccf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams['figure.figsize'] = (11, 4)
np.random.seed(42)

# === Generar serie sintética ===
n = 365 * 3
idx = pd.date_range('2022-01-01', periods=n, freq='D')
t = np.arange(n)

# Componentes
trend     = 100 + 0.05 * t
season_w  = 8  * np.sin(2 * np.pi * t / 7)            # ciclo semanal
season_y  = 25 * np.sin(2 * np.pi * (t - 80) / 365)   # ciclo anual
temp      = 15 + 10 * np.sin(2 * np.pi * (t - 100) / 365) + np.random.normal(0, 2, n)
effect_T  = -1.5 * (temp - 15)                         # más frío -> más demanda
noise     = np.random.normal(0, 4, n)

y = trend + season_w + season_y + effect_T + noise

df = pd.DataFrame({'demanda': y, 'temp': temp}, index=idx)
print(df.head())
df['demanda'].plot(title='Demanda diaria (serie sintética)');

## 1. EDA temporal

Resample a frecuencia semanal y media móvil de 30 días para ver tendencia.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
df['demanda'].plot(ax=ax, alpha=0.4, label='diaria')
df['demanda'].rolling(30).mean().plot(ax=ax, lw=2, label='rolling 30d')
df['demanda'].resample('W').mean().plot(ax=ax, lw=1.5, label='semanal', linestyle='--')
ax.legend(); ax.set_title('Demanda: original vs suavizada');

## 2. Descomposición STL

STL separa **tendencia + estacionalidad + residuo**. Con `period=7` aislamos el ciclo semanal.

In [ ]:
stl = STL(df['demanda'], period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(11, 8)
plt.suptitle('Descomposición STL (period=7)', y=1.02);

## 3. Estacionariedad — Test ADF

H₀: la serie tiene raíz unitaria (NO estacionaria). Si `p < 0.05` rechazamos H₀.

In [ ]:
def adf_report(serie, nombre):
    stat, p, *_ = adfuller(serie.dropna())
    veredicto = 'ESTACIONARIA' if p < 0.05 else 'NO estacionaria'
    print(f'{nombre:25s} | ADF={stat:7.3f} | p={p:.4f} | {veredicto}')

adf_report(df['demanda'],            'Serie original')
adf_report(df['demanda'].diff(),     'Diff(1)')
adf_report(df['demanda'].diff().diff(7), 'Diff(1) + Diff(7)')

## 4. Correlación: ACF, PACF y CCF

- **ACF(k)**: correlación de la serie con su rezago `k`. Sugiere `q` (MA).
- **PACF(k)**: correlación parcial controlando rezagos intermedios. Sugiere `p` (AR).
- **CCF**: correlación cruzada entre `demanda` y la exógena `temp`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_acf(df['demanda'],          ax=axes[0, 0], lags=40); axes[0, 0].set_title('ACF — original')
plot_pacf(df['demanda'],         ax=axes[0, 1], lags=40); axes[0, 1].set_title('PACF — original')
plot_acf(df['demanda'].diff().dropna(),  ax=axes[1, 0], lags=40); axes[1, 0].set_title('ACF — diff(1)')
plot_pacf(df['demanda'].diff().dropna(), ax=axes[1, 1], lags=40); axes[1, 1].set_title('PACF — diff(1)')
plt.tight_layout();

In [ ]:
# Correlación cruzada demanda vs temperatura
lags = 30
cc = ccf(df['demanda'], df['temp'])[:lags]
plt.figure(figsize=(11, 3))
plt.stem(range(lags), cc, basefmt=' ')
plt.axhline(0, color='k', lw=0.5)
plt.axhline( 2/np.sqrt(len(df)), color='r', linestyle='--', lw=0.8)
plt.axhline(-2/np.sqrt(len(df)), color='r', linestyle='--', lw=0.8)
plt.title('CCF: demanda vs temperatura')
plt.xlabel('Lag (días)'); plt.ylabel('Correlación');

print(f'Correlación contemporánea: {df["demanda"].corr(df["temp"]):.3f}')

## 5. Train / test split temporal

**Nunca** usar `train_test_split` aleatorio en series: rompe la dependencia temporal. Cortamos por fecha.

In [ ]:
split = '2024-09-01'
train = df.loc[:split].iloc[:-1]
test  = df.loc[split:]
print(f'Train: {train.shape[0]} días ({train.index.min().date()} -> {train.index.max().date()})')
print(f'Test:  {test.shape[0]} días ({test.index.min().date()} -> {test.index.max().date()})')

def metricas(y_true, y_pred, nombre):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'{nombre:25s} | MAE={mae:6.2f} | RMSE={rmse:6.2f} | MAPE={mape:5.2f}%')
    return {'modelo': nombre, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

resultados = []

## 6. Modelos progresivos

Mismo dataset, complejidad creciente. Comparamos el error en `test`.

### 6.1 AR(p) puro — solo rezagos propios
`ARIMA(p, 0, 0)` con `p` sugerido por PACF.

In [ ]:
modelo_ar = ARIMA(train['demanda'], order=(7, 0, 0)).fit()
pred_ar = modelo_ar.forecast(steps=len(test))
resultados.append(metricas(test['demanda'], pred_ar, 'AR(7)'))

### 6.2 ARMA(p, q) — agrega media móvil

In [ ]:
modelo_arma = ARIMA(train['demanda'], order=(2, 0, 2)).fit()
pred_arma = modelo_arma.forecast(steps=len(test))
resultados.append(metricas(test['demanda'], pred_arma, 'ARMA(2,2)'))

### 6.3 ARIMA(p, d, q) — diferenciación para estacionariedad

In [ ]:
modelo_arima = ARIMA(train['demanda'], order=(2, 1, 2)).fit()
pred_arima = modelo_arima.forecast(steps=len(test))
resultados.append(metricas(test['demanda'], pred_arima, 'ARIMA(2,1,2)'))

### 6.4 SARIMA — estacionalidad semanal
`(p,d,q)(P,D,Q)m` con `m=7`. Captura el ciclo semanal explícitamente.

In [ ]:
modelo_sarima = SARIMAX(
    train['demanda'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
pred_sarima = modelo_sarima.forecast(steps=len(test))
resultados.append(metricas(test['demanda'], pred_sarima, 'SARIMA(1,1,1)(1,1,1,7)'))

### 6.5 SARIMAX — agrega la exógena `temp`
La X de **ARIMAX/SARIMAX** son variables observadas en el mismo instante que `y`. Para pronosticar necesitamos `temp` también en el horizonte de test (acá la conocemos).

In [ ]:
exog_train = train[['temp']]
exog_test  = test[['temp']]

modelo_sarimax = SARIMAX(
    train['demanda'],
    exog=exog_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
pred_sarimax = modelo_sarimax.forecast(steps=len(test), exog=exog_test)
resultados.append(metricas(test['demanda'], pred_sarimax, 'SARIMAX + temp'))
print('\nCoeficiente de la exógena:')
print(modelo_sarimax.params[['temp']])

### 6.6 SARIMAX con exógenas múltiples
Agregamos *dummies* de día de la semana como exógenas adicionales (además de `temp`).

In [ ]:
def construir_exog(frame):
    dow = pd.get_dummies(frame.index.dayofweek, prefix='dow', drop_first=True).astype(float)
    dow.index = frame.index
    return pd.concat([frame[['temp']], dow], axis=1)

exog_train_m = construir_exog(train)
exog_test_m  = construir_exog(test)

modelo_sarimax_m = SARIMAX(
    train['demanda'],
    exog=exog_train_m,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)
pred_sarimax_m = modelo_sarimax_m.forecast(steps=len(test), exog=exog_test_m)
resultados.append(metricas(test['demanda'], pred_sarimax_m, 'SARIMAX + temp + DOW'))

## 7. Comparación de modelos

In [ ]:
tabla = pd.DataFrame(resultados).set_index('modelo').round(2)
print(tabla)

fig, ax = plt.subplots(figsize=(12, 5))
test['demanda'].plot(ax=ax, label='real', color='black', lw=1.5)
pred_arima.plot(ax=ax, label='ARIMA(2,1,2)',  alpha=0.7)
pred_sarima.plot(ax=ax, label='SARIMA',        alpha=0.7)
pred_sarimax_m.plot(ax=ax, label='SARIMAX+exog', alpha=0.9, lw=2)
ax.legend(); ax.set_title('Pronóstico en test: real vs modelos');

## 8. Diagnóstico de residuos (modelo ganador)

Residuos deben ser **ruido blanco**: sin autocorrelación, media ≈ 0, normales.
- **Ljung‑Box**: H₀ residuos no autocorrelacionados → buscamos `p > 0.05`.

In [ ]:
modelo_sarimax_m.plot_diagnostics(figsize=(12, 8));
plt.tight_layout()

lb = acorr_ljungbox(modelo_sarimax_m.resid, lags=[7, 14, 21], return_df=True)
print('Ljung-Box:')
print(lb.round(4))

## 9. Walk‑forward validation

Entrenar con histórico, predecir 1 paso, agregar la observación real, repetir. Más realista que un único forecast largo.

In [ ]:
historia_y = train['demanda'].copy()
historia_x = exog_train_m.copy()
preds_wf = []

for fecha in test.index[:60]:   # 60 pasos para que no demore demasiado
    m = SARIMAX(
        historia_y, exog=historia_x,
        order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    exog_step = exog_test_m.loc[[fecha]]
    yhat = m.forecast(steps=1, exog=exog_step).iloc[0]
    preds_wf.append(yhat)
    historia_y.loc[fecha] = test['demanda'].loc[fecha]
    historia_x.loc[fecha] = exog_test_m.loc[fecha]

preds_wf = pd.Series(preds_wf, index=test.index[:60])
metricas(test['demanda'].iloc[:60], preds_wf, 'SARIMAX walk-forward');

## Ejercicios

### Ejercicio 1 — Elegir `(p, q)` desde ACF/PACF
Genera una serie `ARIMA(2,1,0)` simulada (te dejo el código). Examina su ACF/PACF tras diferenciar y argumenta qué `(p, q)` elegirías.

In [ ]:
from statsmodels.tsa.arima_process import ArmaProcess
np.random.seed(0)
ar  = np.r_[1, -0.6, -0.3]   # phi_1, phi_2
ma  = np.r_[1]
proceso = ArmaProcess(ar, ma).generate_sample(500).cumsum()
serie_ej = pd.Series(proceso)

# TODO: graficar ACF y PACF de serie_ej.diff().dropna() y responder
#       ¿cuántos picos significativos hay en PACF? ¿y en ACF?

### Ejercicio 2 — SARIMAX sin la exógena correcta
Entrena un SARIMAX **omitiendo** `temp`. Compara su MAPE en test con el SARIMAX completo. ¿Cuánto se pierde por no usar la exógena?

In [ ]:
# TODO: ajustar SARIMAX(order=(1,1,1), seasonal_order=(1,1,1,7)) sin exog
#       y comparar métricas vs modelo_sarimax_m

### Ejercicio 3 — Pronóstico con incertidumbre
Usa `get_forecast` en lugar de `forecast` para obtener intervalos de confianza al 95% y grafícalos sobre el test.

In [ ]:
# TODO: fc = modelo_sarimax_m.get_forecast(steps=len(test), exog=exog_test_m)
#       ic = fc.conf_int(alpha=0.05)
#       graficar fc.predicted_mean y la banda ic

### Ejercicio 4 — Auto ARIMA (opcional)
Instala `pmdarima` (`pip install pmdarima`) y usa `auto_arima` con `seasonal=True, m=7` para que busque la combinación óptima. Compara con tu SARIMA manual.

## Checklist conceptual

- [ ] ¿Qué distingue ACF de PACF, y qué parámetro de ARIMA estima cada una?
- [ ] ¿Por qué no se usa k‑fold estándar en series temporales?
- [ ] ¿Qué condición debe cumplir la serie antes de ajustar AR/ARMA?
- [ ] ¿En qué se diferencia SARIMAX de SARIMA?
- [ ] ¿Qué problema operacional tiene SARIMAX para pronósticos a futuro?
- [ ] ¿Qué buscas en el test de Ljung‑Box sobre los residuos?
- [ ] ¿Qué métrica usarías si las series a comparar tienen escalas muy distintas?